DH演習2026 · 教材 第38版 · 2026-r38 · 2026-10-15

<details><summary>教材の版と相談用情報</summary>

相談時は次の情報を伝えてください。

```text
DH演習2026 · 教材 第38版
教材ID: 2026-r38
教材日付: 2026-10-15
活動: source_trial_record-class3
環境: Colab / Jupyter
ソース: 0168ecfac6edcdfec02b46504654b08d76a08d2f
```

</details>

[同じ版の教材・変更点](https://nlp.lang.osaka-u.ac.jp/dh/practicum/editions.html)


# 第3回：領域を座標で表し、分類する

上から順にセルを実行してください（Shift+Enter）。値を変えたら、そのセルから下を再実行します。

資料：[官報 1902年8月20日](https://dl.ndl.go.jp/pid/2949042/1/5)（国立国会図書館デジタルコレクション）

## 1. 画像を取得する

`region`は切り出す範囲（左,上,幅,高さ）で、原画像のピクセルで指定します。

In [ ]:
#@title 操作：道具を読み込む
from PIL import Image, ImageDraw

In [ ]:
koma = 5
region = "2257,334,1059,1811"
url = f"https://dl.ndl.go.jp/api/iiif/2949042/R{koma:07d}/{region}/1520,/0/default.jpg"
!curl -fso page.jpg {url}
Image.open("page.jpg")

## 2. OCRの枠を下書きにする

[NDLOCR-Lite](https://github.com/ndl-lab/ndlocr-lite/tree/d25e0d415b607ad44459ca6b95c7512a54363935)が検出した行ごとに、種類・左・上・幅・高さ・文字列の先頭を表示します。座標は`page.jpg`の左上からのピクセル数で、`region`の原画像の座標とは基準が違います。初回だけ、準備に約20秒かかります。

In [ ]:
import xml.etree.ElementTree as ET

ocr = "git+https://github.com/ndl-lab/ndlocr-lite@d25e0d415b607ad44459ca6b95c7512a54363935"
!uvx -q --from {ocr} ndlocr-lite --sourceimg page.jpg --output . --viz True

for line in ET.parse("page.xml").iter("LINE"):
    print(*(line.get(name) for name in ("TYPE", "X", "Y", "WIDTH", "HEIGHT")), line.get("STRING", "")[:10])
Image.open("viz_page.jpg")

## 3. 領域を囲んで分類する

`regions`の各行は（分類, 左, 上, 幅, 高さ）です。右下の座標は（左＋幅, 上＋高さ）になります。京都府の報告の横にある日付（本月十日）を、報告に含めるか別の領域にするか決めて、`regions`に加えてください。

In [ ]:
regions = [
    ("見出し", 1300, 60, 105, 400),
    ("報告", 1144, 124, 92, 2465),
]
partner = [
    ("報告", 1095, 124, 141, 2465),
]

marked = Image.open("page.jpg").convert("RGB")
draw = ImageDraw.Draw(marked)
for name, color, rows in (("自分", "blue", regions), ("参考例", "orange", partner)):
    for number, (label, x, y, w, h) in enumerate(rows):
        draw.rectangle((x, y, x + w, y + h), outline=color, width=5)
        draw.text((x, y), str(number), fill=color, font_size=40)
        print(name, number, label, (x, y), (x + w, y + h))
marked

## 4. 参考例と比べる

3.の`partner`は、境界の違う参考例です。他の受講者と組む場合は、その人の`regions`の行に置き換えて実行し直すと、その枠が橙色で重なります。境界や分類が違う箇所から分類の規則を一文にし、別の領域でも試してください。

## 5. 別の資料で試す

IIIFに対応した資料なら、1.の`url`をその資料の画像URLに変えると、同じ手順で領域を囲めます。

## 確認問題

鹿兒島縣の報告を囲む領域を、分類「報告」として`regions`に加えてください。囲むのは県名から始まる一列です。3.のセルを実行し直してから、次のセルで答えを確かめてください。

In [ ]:
#@title 答えを確かめる
line = next(row for row in ET.parse("page.xml").iter("LINE") if row.get("STRING", "").startswith("鹿兒島縣"))
target = tuple(int(line.attrib[name]) for name in ("X", "Y", "WIDTH", "HEIGHT"))
widest = 3 * target[2], 3 * target[3]
missing = "鹿兒島縣の行を囲む「報告」の領域がありません。OCRの表の座標を使ってください"
boxes = [(x, y, w, h) for label, x, y, w, h in regions if label == "報告"]
covering = [box for box in boxes if box[0] <= target[0] and box[1] <= target[1]
            and target[0] + target[2] <= box[0] + box[2] and target[1] + target[3] <= box[1] + box[3]]
if not covering:
    print(f"結果：不合格（{missing}）")
elif all(w > widest[0] or h > widest[1] for x, y, w, h in covering):
    print("結果：不合格（領域が広すぎます。対象だけを囲んでください）")
else:
    print("結果：合格")

## 6. 隣のコマに同じ手順を使う（授業外・任意）

`region`と`regions`を変えずに、この号の隣のコマにも使います。ノートブックは`komas`のコマを一つのフォルダに取得し、一つの命令でまとめてOCRを実行します。表示するのは、コマごとに検出した行数と、各領域に重なる文字列の先頭です。

枠を重ねて確かめるコマを`check`に入れて実行してください。

In [ ]:
komas = [5, 6, 7]
check = 6

!mkdir -p koma koma-ocr
for koma in komas:
    url = f"https://dl.ndl.go.jp/api/iiif/2949042/R{koma:07d}/{region}/1520,/0/default.jpg"
    !curl -fso koma/koma{koma}.jpg {url}
!uvx -q --from {ocr} ndlocr-lite --sourcedir koma --output koma-ocr

for koma in komas:
    detected = list(ET.parse(f"koma-ocr/koma{koma}.xml").iter("LINE"))
    print(f"コマ{koma} {len(detected)}行")
    for number, (label, x, y, w, h) in enumerate(regions):
        text = "".join(row.get("STRING", "") for row in detected
                       if int(row.attrib["X"]) < x + w and x < int(row.attrib["X"]) + int(row.attrib["WIDTH"])
                       and int(row.attrib["Y"]) < y + h and y < int(row.attrib["Y"]) + int(row.attrib["HEIGHT"]))
        print(f"  領域{number} {label}：{text[:12]}")

marked = Image.open(f"koma/koma{check}.jpg").convert("RGB")
draw = ImageDraw.Draw(marked)
for number, (_, x, y, w, h) in enumerate(regions):
    draw.rectangle((x, y, x + w, y + h), outline="blue", width=5)
    draw.text((x, y), str(number), fill="blue", font_size=40)
marked

同じ座標でも、コマが変わると囲む内容が変わります。枠を見て確かめていないコマの文字列を、何を根拠に使いますか。